Installing required dependencies

In [ ]:
!pip install -U transformers peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-4-e4b-it",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

model = PeftModel.from_pretrained(
    base_model,
    "krishnamraja13/gemma-4-e4b-opus46-reasoning"
)

tokenizer = AutoTokenizer.from_pretrained(
    "krishnamraja13/gemma-4-e4b-opus46-reasoning"
)

print("HF adapter load successful")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

...

adapter_model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

HF adapter load successful


Passing same prompt to base model + the PEFT adapter

In [ ]:
messages = [
    {
        "role": "user",
        "content": "A few years ago, Jimin was 16 and his father was 47. If the father is twice Jimin's age this year, how many years have passed?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

inputs = {k: v.to("cuda") for k, v in inputs.items()}

# BASE MODEL OUTPUT
base_out = base_model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

adapter_out = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)
print("========== BASE GEMMA 4 E4B ==========\n")
print(tokenizer.decode(base_out[0], skip_special_tokens=True))

print("\n\n========== YOUR OPUS46 ADAPTER ==========\n")
print(tokenizer.decode(adapter_out[0], skip_special_tokens=True))

========== BASE GEMMA 4 E4B ==========

user
A few years ago, Jimin was 16 and his father was 47. If the father is twice Jimin's age this year, how many years have passed?
model
Here's how to solve the problem step-by-step:

**1. Determine the ages a few years ago:**

* **Jimin's age (then):** 16
* **Father's age (then):** 47

**2. Set up variables for "this year":**

Let $Y$ be the number of years that have passed since the initial situation.

* **Jimin's current age:** $16 + Y$
* **Father's current age:** $47 + Y$

**3. Use the condition given for "this year":**

The problem states: "If the father is twice Jimin's age this year."

$$\text{Father's Current Age} = 2 \times (\text{Jimin's Current Age})$$

$$47 + Y = 2(16 + Y)$$

**4. Solve for Y:**

*


========== YOUR OPUS46 ADAPTER ==========

user
A few years ago, Jimin was 16 and his father was 47. If the father is twice Jimin's age this year, how many years have passed?
model
Here's how to solve the problem step-by-step:

**1. Dete